In [1]:
# Kiểm tra nhanh các thư viện cơ bản (không bắt buộc tất cả phải có)
import sys, json, math, random, os, time
import numpy as np

try:
    import sklearn
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import classification_report, confusion_matrix
    SKLEARN_OK = True
except Exception as e:
    SKLEARN_OK = False
    print("Thiếu scikit-learn. Các phần dùng sklearn sẽ không chạy:", e)

try:
    import networkx as nx
    NETWORKX_OK = True
except Exception as e:
    NETWORKX_OK = False
    print("Thiếu networkx. Các phần đồ thị sẽ không chạy:", e)

# Các phần tùy chọn (có thể không có internet nên sẽ fail ở đây, cứ để False nếu không)
try:
    import torch
    TORCH_OK = True
except Exception as e:
    TORCH_OK = False

try:
    import transformers
    TRANSFORMERS_OK = True
except Exception as e:
    TRANSFORMERS_OK = False

print("SKLEARN_OK =", SKLEARN_OK, "| NETWORKX_OK =", NETWORKX_OK, "| TORCH_OK =", TORCH_OK, "| TRANSFORMERS_OK =", TRANSFORMERS_OK)

SKLEARN_OK = True | NETWORKX_OK = True | TORCH_OK = True | TRANSFORMERS_OK = True


In [2]:
# Prefer tqdm.auto so it picks the right frontend (notebook vs terminal)
try:
    from tqdm.auto import tqdm
except Exception:
    from tqdm import tqdm

# Optional: if you still see the destructor AttributeError during shutdown,
# uncomment the monkeypatch below as a last resort (it silences the message):
# import tqdm as _tqdm
# _tqdm.tqdm.__del__ = lambda self: None

## 10) CLIP – Ứng dụng: **Tìm ảnh theo văn bản (Image Search) & Zero-shot Classification** (demo)

**Bối cảnh:** Bạn có một thư mục ảnh sản phẩm, muốn **gõ từ khóa** (ví dụ: "điện thoại đen viền mỏng") để tìm ảnh phù hợp nhanh.

Ta làm 2 hướng:
1. **Mô phỏng ý tưởng CLIP (ngoại tuyến):** trích xuất *đặc trưng ảnh* đơn giản (histogram màu + HOG nhẹ nếu có),
   trích xuất *đặc trưng văn bản* bằng TF-IDF. Quy về cùng không gian qua chuẩn hóa & cosine similarity → **tìm ảnh gần nhất**.
   (Đây không phải CLIP thật, nhưng tái hiện workflow tìm kiếm đa phương thức.)

2. **Tùy chọn (cần internet):** Dùng CLIP pre-trained (`openai/clip-vit-base-patch32`) từ `transformers` để embed ảnh & text,
   sau đó tìm kiếm bằng cosine giống paper gốc.

In [3]:
# Mô phỏng CLIP: tạo 1 "bộ sưu tập" ảnh giả + caption, sau đó truy vấn bằng văn bản
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Giả lập 6 ảnh sản phẩm với 'mô tả' (caption) ngắn
image_ids = [f"img_{i}.jpg" for i in range(6)]
captions = [
    "điện thoại đen viền mỏng camera kép",
    "tai nghe bluetooth màu trắng chống ồn",
    "laptop xám mỏng nhẹ màn 14 inch",
    "điện thoại xanh dương ba camera",
    "máy ảnh đen ống kính lớn",
    "điện thoại đen viền dày pin trâu"
]

# Text encoder (mô phỏng) bằng TF-IDF
vectorizer = TfidfVectorizer(ngram_range=(1,2))
text_embs = vectorizer.fit_transform(captions)   # shape [6, V]

def query_search(query, topk=3):
    q = vectorizer.transform([query])
    sims = cosine_similarity(q, text_embs)[0]
    idx = np.argsort(-sims)[:topk]
    return [(image_ids[i], captions[i], float(sims[i])) for i in idx]

print("Query: 'điện thoại đen viền mỏng'")
for iid, cap, sc in query_search("điện thoại đen viền mỏng"):
    print(f"- {iid:8s} | {cap:45s} | sim={sc:.3f}")

Query: 'điện thoại đen viền mỏng'
- img_0.jpg | điện thoại đen viền mỏng camera kép           | sim=0.777
- img_5.jpg | điện thoại đen viền dày pin trâu              | sim=0.527
- img_3.jpg | điện thoại xanh dương ba camera               | sim=0.201


In [4]:
# %pip install -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
# %pip install -U transformers pillow


In [6]:
USE_REAL_CLIP = True  # bật True nếu đã cài thành công

if USE_REAL_CLIP:
    import torch
    from PIL import Image
    from transformers import CLIPProcessor, CLIPModel

    model_name = "openai/clip-vit-base-patch32"
    model = CLIPModel.from_pretrained(model_name)
    processor = CLIPProcessor.from_pretrained(model_name)

    # Ví dụ: 3 ảnh (bạn thay bằng ảnh thật của bạn)
    imgs = [Image.fromarray(np.random.randint(0,255,(224,224,3),dtype=np.uint8)) for _ in range(3)]
    texts = ["điện thoại đen", "tai nghe trắng", "laptop mỏng nhẹ"]

    inputs = processor(text=texts, images=imgs, return_tensors="pt", padding=True)
    with torch.no_grad():
        out = model(**inputs)
        i_emb = out.image_embeds / out.image_embeds.norm(p=2, dim=-1, keepdim=True)
        t_emb = out.text_embeds / out.text_embeds.norm(p=2, dim=-1, keepdim=True)
        sims = (t_emb @ i_emb.T).cpu().numpy()  # [len(texts), len(imgs)]

    print("Ma trận độ tương đồng (text x image):\n", np.round(sims,3))
else:
    print("Để chạy CLIP thật: bật USE_REAL_CLIP=True sau khi đã cài torch/transformers.")


Exception ignored in: <function tqdm.__del__ at 0x0000026FB49AA980>
Traceback (most recent call last):
  File "c:\Users\CaRot\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\std.py", line 1148, in __del__
    self.close()
  File "c:\Users\CaRot\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\notebook.py", line 279, in close
    self.disp(bar_style='danger', check_delay=False)
    ^^^^^^^^^
AttributeError: 'tqdm' object has no attribute 'disp'


Ma trận độ tương đồng (text x image):
 [[0.203 0.207 0.209]
 [0.2   0.203 0.206]
 [0.221 0.223 0.229]]
